# YawRateSineScenario 디버그

**목적**: `generate()` 가 만드는 시나리오 출력값(시간 배열, 요레이트 지령 파형, 윈도우 함수 등)을 시각적으로 확인한다.

## 1. 환경 설정

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

ROOT_DIR = Path().resolve().parents[3]
sys.path.insert(0, str(ROOT_DIR))

from vehicle_sim.scenarios.yaw_rate_sine import generate, YawRateSineScenario

print(f"ROOT_DIR: {ROOT_DIR}")

## 2. 기본 시나리오 생성 및 메타 정보 확인

In [ ]:
dt = 0.001  # [s]

scenario = generate(dt=dt)

t            = scenario['time']
yaw_rate_ref = scenario['yaw_rate_ref']
target_mps   = scenario['target_mps']

print(f"steps      : {len(t)}")
print(f"dt         : {dt*1000:.1f} ms")
print(f"t_end      : {t[-1]:.3f} s")
print(f"target_mps : {target_mps:.3f} m/s  ({target_mps*3.6:.1f} kph)")
print(f"amp        : {np.max(np.abs(yaw_rate_ref)):.4f} rad/s")
print(f"yaw_rate_ref shape: {yaw_rate_ref.shape}")

## 3. 요레이트 지령 파형 시각화

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
fig.suptitle('YawRateSineScenario — Default Parameters', fontsize=13, fontweight='bold')

axes[0].plot(t, yaw_rate_ref, linewidth=1.5, label='yaw_rate_ref')
axes[0].axhline(0, color='k', linewidth=0.6, alpha=0.4)
axes[0].set_ylabel('Yaw Rate [rad/s]')
axes[0].set_title('Yaw Rate Reference Waveform')
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].plot(t, np.rad2deg(yaw_rate_ref), linewidth=1.5, color='tab:orange', label='yaw_rate_ref')
axes[1].axhline(0, color='k', linewidth=0.6, alpha=0.4)
axes[1].set_ylabel('Yaw Rate [deg/s]')
axes[1].set_xlabel('Time [s]')
axes[1].set_title('Yaw Rate Reference Waveform (deg/s)')
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()

## 4. 윈도우 함수 확인 (ramp_time 효과)

In [ ]:
start_delay = 1.0
ramp_time   = 1.2
amp         = 0.05
freq_hz     = 0.25

mask_ramp = (t >= start_delay) & (t <= start_delay + ramp_time)
t_ramp    = t[mask_ramp] - start_delay
window    = 0.5 * (1.0 - np.cos(np.pi * t_ramp / ramp_time))

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=False)
fig.suptitle('Window Function (Half-Cosine Ramp)', fontsize=13, fontweight='bold')

axes[0].plot(t_ramp, window, linewidth=2, label='window(t)')
axes[0].set_ylabel('Amplitude Scale [-]')
axes[0].set_xlabel('t_rel [s] (from start_delay)')
axes[0].set_title(f'ramp_time = {ramp_time} s')
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].plot(t, yaw_rate_ref, linewidth=1.2, alpha=0.5, color='gray', label='full signal')
axes[1].plot(t[mask_ramp], yaw_rate_ref[mask_ramp], linewidth=2, color='tab:red', label='ramp region')
axes[1].axvline(start_delay, color='k', linestyle='--', linewidth=0.8, label=f'start_delay={start_delay}s')
axes[1].axvline(start_delay + ramp_time, color='tab:green', linestyle='--', linewidth=0.8, label=f'ramp_end={start_delay+ramp_time}s')
axes[1].set_ylabel('Yaw Rate [rad/s]')
axes[1].set_xlabel('Time [s]')
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()

## 5. 파라미터 변화 비교

In [ ]:
cases = [
    dict(amp=0.05, freq_hz=0.25, label='default (amp=0.05, freq=0.25Hz)'),
    dict(amp=0.10, freq_hz=0.25, label='amp=0.10, freq=0.25Hz'),
    dict(amp=0.05, freq_hz=0.50, label='amp=0.05, freq=0.50Hz'),
    dict(amp=0.10, freq_hz=0.50, label='amp=0.10, freq=0.50Hz'),
]

fig, ax = plt.subplots(figsize=(13, 5))
fig.suptitle('Parameter Sweep Comparison', fontsize=13, fontweight='bold')

for c in cases:
    s = generate(dt=dt, amp=c['amp'], freq_hz=c['freq_hz'])
    ax.plot(s['time'], s['yaw_rate_ref'], linewidth=1.5, label=c['label'])

ax.axhline(0, color='k', linewidth=0.6, alpha=0.4)
ax.set_ylabel('Yaw Rate [rad/s]')
ax.set_xlabel('Time [s]')
ax.grid(True, alpha=0.3)
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

## 6. `yaw_rate_ref(i)` 메서드 확인

In [ ]:
# 시뮬레이션 루프에서 실제로 사용하는 방식 확인
print('인덱스별 yaw_rate_ref(i) 확인:')
for i in [0, 500, 1000, 1500, 2000, 3000, 4000]:
    val = scenario.yaw_rate_ref(i)
    print(f'  i={i:5d}  t={scenario["time"][i]:.3f}s  yaw_rate_ref={val:+.5f} rad/s  ({np.rad2deg(val):+.3f} deg/s)')